In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir(r"E:\retailpulse")

df = pd.read_csv("data/processed/master.csv",
                 parse_dates=["order_date"])

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Date range: {df['order_date'].min()} → {df['order_date'].max()}")
print(f"Customers: {df['customer_id_unique'].nunique()}")
print(f"Categories: {df['category'].nunique()}")
print(f"\nSample:")
print(df.head(3).to_string())

Shape: (110189, 18)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_date', 'order_delivered_customer_date', 'customer_id_unique', 'customer_state', 'order_payment', 'category', 'year', 'month', 'week', 'revenue']
Date range: 2016-09-15 12:16:38 → 2018-08-29 15:00:37
Customers: 93350
Categories: 71

Sample:
                           order_id  order_item_id                        product_id                         seller_id  shipping_limit_date  price  freight_value                       customer_id          order_date order_delivered_customer_date                customer_id_unique customer_state  order_payment         category  year  month  week  revenue
0  00010242fe8c5a6d1ba2dd792cb16214              1  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202  2017-09-19 09:45:35   58.9          13.29  3ce436f183e68e07877b285a838db11a 2017-09-13 08:59:02           2017-09-20 23:43:48  8

In [2]:
# Monthly revenue trend
monthly = (df.groupby(df["order_date"].dt.to_period("M"))
             .agg(revenue=("revenue","sum"),
                  orders=("order_id","nunique"),
                  customers=("customer_id_unique","nunique"))
             .reset_index())
monthly["order_date"] = monthly["order_date"].astype(str)

fig = px.line(monthly, x="order_date", y="revenue",
              title="Monthly Revenue Trend (2016-2018)",
              labels={"order_date":"Month","revenue":"Revenue (BRL)"})
fig.show()

print(f"Total Revenue:   BRL {df['revenue'].sum():,.0f}")
print(f"Total Orders:    {df['order_id'].nunique():,}")
print(f"Total Customers: {df['customer_id_unique'].nunique():,}")
print(f"Avg Order Value: BRL {df.groupby('order_id')['revenue'].sum().mean():,.2f}")

Total Revenue:   BRL 15,418,395
Total Orders:    96,470
Total Customers: 93,350
Avg Order Value: BRL 159.83


In [3]:
# Top 10 categories by revenue
cat_rev = (df.groupby("category")["revenue"]
             .sum()
             .sort_values(ascending=False)
             .head(10)
             .reset_index())

fig = px.bar(cat_rev, x="revenue", y="category",
             orientation="h",
             title="Top 10 Categories by Revenue",
             labels={"revenue":"Revenue (BRL)","category":"Category"},
             color="revenue",
             color_continuous_scale="Blues")
fig.update_layout(yaxis={"categoryorder":"total ascending"})
fig.show()

print("Top 5 Categories:")
print(cat_rev.head().to_string(index=False))

Top 5 Categories:
             category    revenue
        health_beauty 1412089.53
        watches_gifts 1264016.98
       bed_bath_table 1225209.26
       sports_leisure 1118062.91
computers_accessories 1032603.65


In [4]:
# How many orders per customer?
orders_per_customer = (df.groupby("customer_id_unique")["order_id"]
                         .nunique()
                         .reset_index()
                         .rename(columns={"order_id":"num_orders"}))

print("Orders per customer distribution:")
print(orders_per_customer["num_orders"].value_counts().head(10))

repeat_customers = (orders_per_customer["num_orders"] > 1).sum()
total_customers  = len(orders_per_customer)
print(f"\nRepeat customers: {repeat_customers} "
      f"({repeat_customers/total_customers*100:.1f}%)")
print(f"One-time buyers:  "
      f"{total_customers - repeat_customers} "
      f"({(total_customers-repeat_customers)/total_customers*100:.1f}%)")

Orders per customer distribution:
num_orders
1     90549
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64

Repeat customers: 2801 (3.0%)
One-time buyers:  90549 (97.0%)


In [5]:
# Monthly demand pattern
monthly_avg = (df.groupby("month")["revenue"]
                 .mean()
                 .reset_index())

fig = px.bar(monthly_avg, x="month", y="revenue",
             title="Average Revenue by Month",
             labels={"month":"Month","revenue":"Avg Revenue (BRL)"},
             color="revenue",
             color_continuous_scale="Oranges")
fig.show()

# Day of week pattern
df["dayofweek"] = df["order_date"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday",
             "Thursday","Friday","Saturday","Sunday"]
dow = (df.groupby("dayofweek")["revenue"]
         .mean()
         .reindex(dow_order)
         .reset_index())

fig2 = px.bar(dow, x="dayofweek", y="revenue",
              title="Average Revenue by Day of Week",
              labels={"dayofweek":"Day","revenue":"Avg Revenue"})
fig2.show()

In [6]:
monthly.to_csv("data/processed/monthly_sales.csv", index=False)
cat_rev.to_csv("data/processed/category_revenue.csv", index=False)

print("✅ EDA outputs saved!")
print("\n📊 KEY BUSINESS INSIGHTS:")
print("="*45)
print(f"1. Total revenue: BRL {df['revenue'].sum():,.0f}")
print(f"2. Peak month:    "
      f"{monthly.loc[monthly['revenue'].idxmax(),'order_date']}")
print(f"3. Top category:  {cat_rev.iloc[0]['category']}")
print(f"4. Repeat buyers: "
      f"{repeat_customers/total_customers*100:.1f}% of customers")
print("="*45)

✅ EDA outputs saved!

📊 KEY BUSINESS INSIGHTS:
1. Total revenue: BRL 15,418,395
2. Peak month:    2017-11
3. Top category:  health_beauty
4. Repeat buyers: 3.0% of customers
